# YOLO 사전학습 모델 기반 Fine Tuning

## 1. 기본 설정 (GPU 런타임 확인, Ultralytics 확인)

### 1. GPU 확인

In [ ]:
!nvidia-smi

### 2. Ultralytics 등 패키지 설치 및 import & 버전 확인

In [ ]:
!pip install -q ultralytics==8.4.183   # same version as the edge device

import shutil, yaml, torch, ultralytics
from pathlib import Path
from collections import Counter
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)

## 2. 사전학습 모델 로드 테스트

### 1. 예제 데이터로 학습

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # 사전학습 가중치 자동 다운로드
model.train(
    data="coco8.yaml",       # 예제 데이터 자동 다운로드
    epochs=1,
    imgsz=640,
    batch=8,
    project="/content/runs",
    name="smoke",
    exist_ok=True,           # 다시 돌려도 smoke2, smoke3로 늘어나지 않게
)

### 2. 결과 파일 확인

In [ ]:
!ls /content/runs/smoke /content/runs/smoke/weights

import pandas as pd
df = pd.read_csv("/content/runs/smoke/results.csv")
print(df.columns.tolist())

### 3. 저장된 모델로 추론

In [ ]:
m = YOLO("/content/runs/smoke/weights/best.pt")
r = m.predict("https://ultralytics.com/images/bus.jpg", conf=0.25, verbose=False)[0]
print("classes in model:", len(m.names))
print("detections:", [(m.names[int(c)], round(float(s), 2))
                      for c, s in zip(r.boxes.cls, r.boxes.conf)])

## 3. Drive 마운트 및 데이터 정리

**!!실행 전에 아래 셀에 경로 설정!!**

In [ ]:
# ===== Settings =====
SMOKE_TEST = True     # True: coco8 example data / False: our dataset

DATA_SRC = "<Google Drive의 dataset 폴더 혹은 .zip 파일 경로>"
CLASS_NAMES = ["<class0>", "<class1>", "<class2>"]   # 라벨링된 순서에 맞게 클래스 이름 지정

RUN_ROOT = Path("<현재 run을 저장할 Google Drive 경로>")
MODEL = "yolo11n.pt"   # backbone 모델 지정
RUN_NAME = "smoke_n"   # run name 지정!! e.g. <date>_<data>_<model>
EPOCHS = 5
# ====================

In [ ]:
## Drive 마운트

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
## 공유 문서 접속

!ls "/content/drive/MyDrive/Physical_AI_로봇_SW_개발/OnDevice_AI_TeamProject/LiDAR_Blind_Spot"

전제: 데이터 구조가 images/{train,val}, labels/{train,val} 구조라고 가정

In [ ]:
## 데이터 복사, data.yaml 생성, 검사

import shutil, yaml
from pathlib import Path
from collections import Counter


if SMOKE_TEST:
    d = check_det_dataset("coco8.yaml")   # 테스트 시에는 예시 자료 이용
    SRC, CLASS_NAMES = Path(d["path"]), d["names"]
else:
    SRC = Path(DATA_SRC)

DATA_DIR = Path("/content/data")
IMG_EXT = {".jpg", ".jpeg", ".png"}

# 1) 복사 (매번 새로)
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
if SRC.suffix == ".zip":
    shutil.unpack_archive(SRC, DATA_DIR)
else:
    shutil.copytree(SRC, DATA_DIR)

# 2) data.yaml 생성
SPLITS = ["train", "val"] + (["test"] if (DATA_DIR / "images" / "test").exists() else [])
cfg = {"path": str(DATA_DIR), "train": "images/train",
       "val": "images/val", "names": CLASS_NAMES}
if "test" in SPLITS:
    cfg["test"] = "images/test"
with open(DATA_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

# 3) 검사
nc = len(CLASS_NAMES)
for split in SPLITS:
    imgs = [p for p in (DATA_DIR / "images" / split).glob("*")
            if p.suffix.lower() in IMG_EXT]
    stems = {p.stem for p in imgs}
    n_neg, bad, counts = 0, [], Counter()
    for img in imgs:
        lbl = DATA_DIR / "labels" / split / (img.stem + ".txt")
        if not lbl.exists() or lbl.stat().st_size == 0:
            n_neg += 1
            continue
        for i, line in enumerate(lbl.read_text().splitlines(), 1):
            p = line.split()
            ok = (len(p) == 5 and p[0].isdigit() and int(p[0]) < nc
                  and all(0 <= float(v) <= 1 for v in p[1:]))
            if ok:
                counts[int(p[0])] += 1
            else:
                bad.append(f"{lbl.name}:{i}")
    orphans = [l.name for l in (DATA_DIR / "labels" / split).glob("*.txt")
               if l.stem not in stems]
    print(f"[{split}] images {len(imgs)}, no-label {n_neg}, "
          f"bad lines {len(bad)}, orphan labels {len(orphans)}")
    print("  boxes:", {CLASS_NAMES[k]: v for k, v in sorted(counts.items())})
    if bad: print("  bad:", bad[:5])
    if orphans: print("  orphans:", orphans[:5])

검사 항목과 이유:

- 클래스별 박스 수: 가장 위험한 실수는 라벨 도구의 id 순서(예: 0=cup)와 CLASS_NAMES 순서(예: 0=box)가 다른 경우다. 이 실수는 오류 없이 학습되고, 모델이 이름만 바뀐 채 결과를 낸다. 박스 수를 B가 알려 준 클래스별 수와 맞춰 보면 잡을 수 있다.
- no-label: 빈 바닥 사진(약 100장)이 여기에 잡혀야 한다. 기대보다 많으면 라벨이 빠진 사진이 섞인 것이다.
- bad lines: 좌표가 0~1 범위를 벗어나거나 id가 범위 밖이면, 내보내기 형식이 잘못된 것이다.
- orphan labels: 사진 없이 라벨만 있는 파일이다. 파일 이름 규칙이 어긋났다는 신호다.

## 4. 학습 및 결과 저장

설계
- 결과 폴더는 개인 Drive에 둔다. 공유 폴더(04_결과)에는 D4에서 확정한 모델만 올린다. 시험 학습까지 공유 폴더에 쌓지 않는다.
- Ultralytics는 epoch마다 weights/last.pt를 덮어 쓴다. 이것이 Drive에 남아 있으면 resume=True로 이어서 학습 가능하다.
- 실행 설정은 Ultralytics가 run 폴더의 args.yaml에 자동으로 남긴다.

In [ ]:
## 학습


model = YOLO(MODEL)
model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=640,      # camera is 640x480
    batch=16,
    patience=20,
    seed=0,
    workers=2,
    project=str(RUN_ROOT),
    name=RUN_NAME,  # creates RUN_NAME2, ... instead of overwriting
)

In [ ]:
## 학습 끊겼을 경우 이어서 학습

last = RUN_ROOT / RUN_NAME / "weights" / "last.pt"
YOLO(str(last)).train(resume=True)

resume은 args.yaml에 적힌 설정 그대로 다시 사용한다. 데이터 경로도 동일하게 '/content/data/data.yaml' 로 저장되어 있는데, 세션이 끊기면 이 폴더는 사라지므로, 이어서 학습 전에 데이터 세팅을 먼저 다시 실행해야 한다.

In [ ]:
## val로 클래스별 성능 보기

best = YOLO(str(RUN_ROOT / RUN_NAME / "weights" / "best.pt"))
metrics = best.val(data=str(DATA_DIR / "data.yaml"), split="val",
                   imgsz=640, batch=16,
                   project=str(RUN_ROOT), name=RUN_NAME + "_val")
print("mAP50", round(metrics.box.map50, 3), "mAP50-95", round(metrics.box.map, 3))